## Setup (Kaggle)

This notebook is **Kaggle-only**. It assumes the project code is already available in the Kaggle session (for example, uploaded as a Kaggle Dataset).

In [ ]:
# Kaggle-only: attach the repo code as a Kaggle Dataset (or upload files).
# This cell finds the project root so installs/imports work reliably.

from __future__ import annotations

import sys
from pathlib import Path


def _find_project_root() -> Path:
    """Find a folder that looks like this repo (has requirements.txt + src/)."""
    here = Path(".").resolve()
    if (here / "requirements.txt").is_file() and (here / "src").is_dir():
        return here

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for p in kaggle_input.rglob("requirements.txt"):
            root = p.parent
            if (root / "src").is_dir():
                return root

    raise FileNotFoundError(
        "Could not find project root (requirements.txt + src/). "
        "Attach the repo as a Kaggle Dataset or upload the repo files into the session."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print("project root:", PROJECT_ROOT)
print("requirements:", (PROJECT_ROOT / "requirements.txt"))


Cloning into '/kaggle/working/project'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 122 (delta 44), reused 106 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 52.35 KiB | 4.76 MiB/s, done.
Resolving deltas: 100% (44/44), done.


## Inputs

In the Kaggle notebook sidebar, use **Add data** and attach:

- Hotels CSV
- World cities (`worldcitiespop.csv` or equivalent)
- Processed joined Parquet: `hotels_with_cities*.parquet`


## Locate datasets

In Kaggle, attached datasets appear under `/kaggle/input/...`.


In [ ]:
from pathlib import Path

# Kaggle-only: list attached datasets so you can locate file paths.
root = Path("/kaggle/input")
print("/kaggle/input exists:", root.is_dir())

# Optional: update these if you want explicit file existence checks.
HOTELS = None
WORLD = None

# Example usage (edit to match your Kaggle mounts):
# HOTELS = Path("/kaggle/input/<dataset-folder>/hotels.csv")
# WORLD  = Path("/kaggle/input/<dataset-folder>/worldcitiespop.csv")

if HOTELS is not None:
    print("hotels exists:", HOTELS.is_file(), HOTELS)
if WORLD is not None:
    print("world exists:", WORLD.is_file(), WORLD)


hotels exists: False /kaggle/input/datasets/raj713335/tbo-hotels-dataset/hotels.csv
world exists: False /kaggle/input/datasets/organizations/max-mind/world-cities-database/worldcitiespop.csv


In [2]:
# Install project requirements if needed.
# Prefer requirements.txt next to the attached code; otherwise search /kaggle/input.

from pathlib import Path

req = PROJECT_ROOT / "requirements.txt"
if not req.is_file():
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.is_dir():
        matches = sorted(kaggle_root.rglob("requirements.txt"))
        if matches:
            req = matches[0]

if not req.is_file():
    raise FileNotFoundError(
        f"requirements.txt not found. Looked at {PROJECT_ROOT / 'requirements.txt'} and under /kaggle/input." 
    )

print("installing requirements from:", req)
!pip install -q -r "{req}"

/kaggle/working/project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 2.3 MB/s eta 0:00:00


## Optional: regenerate the processed joined dataset

The processed Parquet exists **because wrangling was already run previously** (cleaning + join + engineered features). In most runs, you should **use the existing processed Parquet** and skip regeneration.

If you do need to regenerate, enable the next cell.


In [ ]:
# Optional regeneration step (disabled by default).
# This is intentionally guarded so the notebook does NOT rerun wrangling/cleaning just to show outputs.

RUN_CLEANING = False

if RUN_CLEANING:
    !python3 scripts/pipeline/run_cleaning.py \
        --output-dir /kaggle/working/processed \
        --format parquet \
        --chunksize 50000 \
        --progress-every 50000
else:
    print("RUN_CLEANING is False — using the existing processed joined dataset.")


Traceback (most recent call last):
  File "/home/pablo/anaconda3/envs/hotel/lib/python3.9/pathlib.py", line 1251, in mkdir
    self._accessor.mkdir(self, mode)
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/hotel_cleaned'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/pablo/anaconda3/envs/hotel/lib/python3.9/pathlib.py", line 1251, in mkdir
    self._accessor.mkdir(self, mode)
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/pablo/hotel-data-analysis/notebooks/../scripts/pipeline/run_cleaning.py", line 83, in <module>
    main()
  File "/home/pablo/hotel-data-analysis/notebooks/../scripts/pipeline/run_cleaning.py", line 68, in main
    written = run_cleaning_pipeline(
  File "/home/pablo/hotel-data-analysis/src/cleaning/hotel_world_clean.py", line 

## Use the processed dataset created above

Now we use the **processed joined Parquet** that already exists because wrangling was already run previously (cleaning + join + engineered features).

**This section outputs:**
- dataset **size** (rows, columns)
- **5-row preview** (what one hotel record looks like)
- **top 12 columns by % missing** (coverage snapshot)
- **distribution of hotel star ratings** (counts per `hotel_star_rating`)


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd


def _find_joined_parquet() -> Path:
    """Kaggle-only: find hotels_with_cities*.parquet under /kaggle/input."""

    kaggle_root = Path("/kaggle/input")
    if not kaggle_root.is_dir():
        raise FileNotFoundError("/kaggle/input not found. This notebook is Kaggle-only.")

    matches = sorted(kaggle_root.rglob("hotels_with_cities*.parquet"))
    if matches:
        return matches[0]

    raise FileNotFoundError(
        "Could not find hotels_with_cities*.parquet under /kaggle/input. "
        "Attach the processed joined Parquet as a Kaggle Dataset (it exists because wrangling was run previously)."
    )


# Now we use the processed dataset created above.
joined = _find_joined_parquet()

# Load the joined, processed table.
df = pd.read_parquet(joined)

# Normalize any alias column names to the canonical schema.
try:
    from src.modeling.feature_matrix import normalize_engineered_column_names

    df = normalize_engineered_column_names(df)
except Exception:
    pass

print("processed Parquet:", joined)
print("rows:", len(df), "cols:", df.shape[1])

display(df.head(5))

na_rate = (df.isna().mean() * 100).sort_values(ascending=False)
display(na_rate.head(12).to_frame(name="% missing"))

if "hotel_star_rating" not in df.columns:
    raise KeyError("Expected column 'hotel_star_rating' not found in processed dataset.")

display(df["hotel_star_rating"].value_counts(dropna=False).sort_index().to_frame(name="count"))


## Modeling (Deliverable Section)

This section trains and compares multiple regression models for hotel rating prediction.

Current supported models in this repo:
- `linear`
- `ridge`
- `lasso`
- `rf` (Random Forest)
- `xgb` (XGBoost)

Evaluation metrics reported by the training script:
- `RMSE`, `MAE`, `R2`
- `|error| <= 0.5` hit rate
- `|error| <= 1.0` hit rate

For faster notebook iteration, start with a sample. For final results, increase sample size or use full data.

In [6]:
!python3 scripts/modeling/train_baseline_model.py --model linear --project-root /kaggle/working/project

joined table: /kaggle/input/datasets/daniahasan3/hotels-with-cities-2-parquet/hotels_with_cities-2.parquet
rows used: 694,365
RMSE: 0.7553
R2:   0.1960
wrote artifacts under /kaggle/working/model_artifacts


In [7]:
!python3 scripts/modeling/train_baseline_model.py --model rf --project-root /kaggle/working/project --out-dir /kaggle/working/model_artifacts_rf

joined table: /kaggle/input/datasets/daniahasan3/hotels-with-cities-2-parquet/hotels_with_cities-2.parquet
rows used: 694,365
RMSE: 0.6531
R2:   0.3989
wrote artifacts under /kaggle/working/model_artifacts_rf


In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("/kaggle/working/project")
RUNS_DIR = Path("/kaggle/working/model_runs")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Use a sample for iteration speed in notebook. Set to None for full dataset.
SAMPLE_ROWS = 100_000
RANDOM_STATE = 42

models_to_run = [
    ("linear", []),
    ("ridge", ["--ridge-alpha", "10"]),
    ("lasso", ["--lasso-alpha", "0.001"]),
    ("rf", ["--rf-estimators", "200", "--rf-max-depth", "20"]),
    ("xgb", ["--xgb-estimators", "300", "--xgb-max-depth", "8", "--xgb-learning-rate", "0.05"]),
]

for model_name, extra_args in models_to_run:
    out_dir = RUNS_DIR / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python",
        "scripts/modeling/train_baseline_model.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--model",
        model_name,
        "--out-dir",
        str(out_dir),
        "--random-state",
        str(RANDOM_STATE),
    ]
    if SAMPLE_ROWS is not None:
        cmd.extend(["--sample", str(SAMPLE_ROWS)])
    cmd.extend(extra_args)

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

print("Done. Metrics files saved under:", RUNS_DIR)

In [ ]:
summary_rows = []
for model_name in ["linear", "ridge", "lasso", "rf", "xgb"]:
    metrics_path = RUNS_DIR / model_name / "metrics.json"
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    summary_rows.append(
        {
            "model": model_name,
            "rows": metrics["rows"],
            "rmse": metrics["rmse"],
            "mae": metrics["mae"],
            "r2": metrics["r2"],
            "within_0_5": metrics["within_0_5"],
            "within_1_0": metrics["within_1_0"],
            "tuned": metrics.get("tuned", False),
        }
    )

results_df = pd.DataFrame(summary_rows).sort_values(by="rmse", ascending=True)
results_df

### Hyperparameter Tuning (Model Assessment)

To satisfy rubric requirements for model assessment and iterative tuning, run tuned versions of selected models.

- `rf` and `xgb` use `RandomizedSearchCV`
- `ridge` uses `RidgeCV`
- scoring metric for tuning: `neg_root_mean_squared_error`

> Tip: keep `--tune-iters` small while iterating, then increase for final run.

In [ ]:
tuned_jobs = [
    ("ridge", ["--tune", "--cv-folds", "3"]),
    ("rf", ["--tune", "--cv-folds", "3", "--tune-iters", "12"]),
    ("xgb", ["--tune", "--cv-folds", "3", "--tune-iters", "12"]),
]

for model_name, tune_args in tuned_jobs:
    out_dir = RUNS_DIR / f"{model_name}_tuned"
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python",
        "scripts/modeling/train_baseline_model.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--model",
        model_name,
        "--out-dir",
        str(out_dir),
        "--random-state",
        str(RANDOM_STATE),
    ]
    if SAMPLE_ROWS is not None:
        cmd.extend(["--sample", str(SAMPLE_ROWS)])
    cmd.extend(tune_args)

    print("Running tuned:", " ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

print("Tuned runs completed.")

In [ ]:
all_rows = []
for p in sorted(RUNS_DIR.glob("*/metrics.json")):
    m = json.loads(p.read_text(encoding="utf-8"))
    all_rows.append(
        {
            "run": p.parent.name,
            "model": m["model"],
            "rmse": m["rmse"],
            "mae": m["mae"],
            "r2": m["r2"],
            "within_0_5": m["within_0_5"],
            "within_1_0": m["within_1_0"],
            "tuned": m.get("tuned", False),
            "best_params": m.get("best_params", {}),
        }
    )

comparison_df = pd.DataFrame(all_rows).sort_values(by="rmse", ascending=True)
comparison_df

### Hypothesis Testing (Simulation-based)

We run permutation-based hypothesis tests on linear coefficients to evaluate whether observed relationships are stronger than chance.

Tested features:
- `city_population`
- `attractions_count`

Interpretation notes:
- p-value < 0.05 suggests statistically significant association under the model assumptions.
- This is correlation evidence, not causal proof.

In [ ]:
hypo_out = RUNS_DIR / "hypothesis_tests.json"
cmd = [
    "python",
    "scripts/modeling/permutation_hypothesis_tests.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--out",
    str(hypo_out),
    "--n-permutations",
    "500",
    "--random-state",
    str(RANDOM_STATE),
]
if SAMPLE_ROWS is not None:
    cmd.extend(["--sample", str(SAMPLE_ROWS)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

hypo = json.loads(hypo_out.read_text(encoding="utf-8"))
pd.DataFrame(hypo["tests"]).T

### Modeling Takeaways

- Compare models by RMSE/MAE first; use R² and hit rates as supporting metrics.
- Use tuned model results to justify final model choice.
- For interpretability, report regularized linear coefficients and permutation-test evidence.
- For predictive performance, tree-boosting models (e.g., XGBoost) are usually strongest on this tabular setup.
- Clearly separate predictive importance from causal claims in final discussion.